In [ ]:
###
# セットアップ
###

import sys
import os
from pathlib import Path

from google.colab import drive
drive.mount("/content/drive")

ROOT_PATH = Path("/content/drive/MyDrive/cnn-hands-on")

if str(ROOT_PATH) not in sys.path:
    sys.path.append(str(ROOT_PATH))

LOCAL_DATA_DIR = Path("/content/data/cats_vs_dogs")

if not LOCAL_DATA_DIR:
    !unzip -q /content/drive/Mydrive/cnn-hands-on/data/cats_vs_dogs.zip -d /content/data/cats_vs_dogs

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib.pyplot as plt
import pandas as pd
from tqdm import tqdm
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms


# GPUが使える場合はGPUを、使えない場合はCPUを使用する設定
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"使用するデバイス: {device}")

In [ ]:
_FRAC_MAP = {"small":0.2, "medium":0.5, "large":1.0}

# ToDo:データセット実装
class CDDataset(Dataset):
  def __init__(self, df, data_dir, transform=None):
    self.df = df.reset_index(drop=True)
    self.data_dir = data_dir
    self.transform = transform
  
  def __len__(self):
    return len(self.df)
  
  def __getitem__(self, idx):
    row = self.df.iloc[idx]
    image = Image.open(os.path.join(self.data_dir, row["filepath"])).convert("RGB")
    if self.transform:
      image = self.transform(image)
    return image, row["label"]

# ToDo:データローダ実装
def get_dc_dataloaders(data_dir, data_size="small", batch_size=32):
  frac = _FRAC_MAP.get(data_size)
  if frac is None:
    raise ValueError(
        "FRAC ERROR"
    )
  
  df = pd.read_csv(os.path.join(data_dir, "labels.csv"))
  transform = transforms.Compose(
      [transforms.Resize((128,128)), transforms.ToTensor()]
  )

  def make_loader(split, shuffle):
    dataset = CDDataset(
        df[df["split"] == split].sample(frac=frac, random_state=61),
        data_dir,
        transform,
    )
    kwargs = {"num_workers": 2, "pin_memory": True} if shuffle else {}
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, **kwargs)
  
  return (
      make_loader("train", True),
      make_loader("val", False),
      make_loader("test", False),
  )

In [ ]:
train_loader, val_loader, test_loader = get_dc_dataloaders(
    data_dir=LOCAL_DATA_DIR,
    data_size="small",
    batch_size=32
)

In [ ]:
# ToDo:CNNの実装
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1)

        self.fc1 = nn.Linear(in_features=32 * 32 * 32, out_features=128)
        self.fc2 = nn.Linear(in_features=128, out_features=2)
    
    def forward(self, x):
        x = self.conv1(x)
        x = F.relu(x)
        x = F.max_pool2d(x, kernel_size=2)

        x = self.conv2(x)
        x = F.relu(x)
        x = F.max_pool2d(x, kernel_size=2)

        x = torch.flatten(x, 1)

        x = self.fc1(x)
        x = F.relu(x)
        x = self.fc(x)

        return x

model = SimpleCNN().to(device)
print(model)

In [ ]:
# 学習の前準備

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

num_epochs = 20
train_loss_list, val_loss_list, val_acc_list = [], [], []

best_val_loss = float("inf")

os.makedirs(ROOT_PATH / "models", exist_ok=True)
save_path = ROOT_PATH / "models" / "06_cnn.pth"

In [ ]:
# 学習ループ実装

print("学習開始")
for epoch in range(num_epochs):
    # --- Train ---
    model.train()
    running_train_loss = 0.0
    
    train_bar = tqdm(train_loader, desc=f"Epoch [{epoch+1}/{num_epochs}] Train ", leave=False)
    # バッチごとにデータを取り出して学習
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        # ToDo:ステップの更新
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_train_loss += loss.item()

        train_bar.set_postfix({"loss": f"{loss.item():.4f}"})
        
    # 1エポック分の平均Lossを記録
    epoch_train_loss = running_train_loss / len(train_loader)
    train_loss_list.append(epoch_train_loss)
    
    # --- Validation ---
    model.eval()
    running_val_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        val_bar = tqdm(val_loader, desc=f"Epoch [{epoch+1}/{num_epochs}] Val ", leave=False)
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_val_loss += loss.item()

            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    epoch_val_loss = running_val_loss / len(val_loader)
    epoch_val_acc = 100 * correct / total

    val_loss_list.append(epoch_val_loss)
    val_acc_list.append(epoch_val_acc)

    # ToDo:モデルの保存
    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        torch.save(model.state_dict(), save_path)
        mark = "Best Model Saved"
    else:
        mark = ""

    print(f"Epoch [{epoch+1}/{num_epochs}] | "
          f"Train Loss: {epoch_train_loss:.4f} | "
          f"Val Loss: {epoch_val_loss:.4f} | "
          f"Val Acc: {epoch_val_acc:.2f}% {mark}")
 
print("学習完了")

In [ ]:
###
# 5. 結果の描画 (グラフ)
###

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# --- Loss（誤差）の推移 ---
ax1.plot(train_loss_list, label='Train Loss', color='blue', marker='o')
ax1.plot(val_loss_list, label='Validation Loss', color='orange', marker='o')


ax1.set_title('Loss Curve')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.legend()
ax1.grid(True)

# --- Accuracy（正答率）の推移 ---
ax2.plot(val_acc_list, label='Validation Accuracy', color='green', marker='o')
ax2.set_title('Accuracy Curve')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()